### XMAC02 - Atividade Avaliativa 2
#### Nome:
#### Nro Matric:

In [68]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from statsmodels.stats import proportion, weightstats
from scipy.stats import norm, f

Para as questões 1 e 2 use o dataset stackoverflow_2024, que contém informações sobre profissionais de computação do mundo todo coletadas no ano de 2024.

In [69]:
df = pd.read_csv('stackoverflow_2024.csv')

df.head()

,response_id,country,age_group,education,employment,remote_work,developer_type,industry,years_code,years_code_pro,...,uses_csharp,uses_sql,uses_r,ai_use_status,ai_sentiment,uses_ai_now,uses_ai_or_plans,is_remote,is_in_person,is_hybrid
0,73,Pakistan,18-24 years old,"Secondary school (e.g. American high school, G...","Employed, full-time;Student, full-time;Indepen...","Hybrid (some remote, some in-person)",Data scientist or machine learning specialist,Software Development,3.0,1.0,...,False,True,True,Yes,Very favorable,True,True,False,False,True
1,380,Turkey,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Employed, full-time",Remote,Data scientist or machine learning specialist,Computer Systems Design and Services,15.0,6.0,...,False,True,False,Yes,Very favorable,True,True,True,False,False
2,386,France,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",Remote,"Developer, back-end",NaN,27.0,17.0,...,False,False,False,"No, and I don't plan to",NaN,False,False,True,False,False
3,390,United States of America,25-34 years old,Some college/university study without earning ...,"Employed, full-time;Student, part-time",Remote,Student,"Transportation, or Supply Chain",7.0,7.0,...,False,True,False,Yes,Very favorable,True,True,True,False,False
4,393,United Kingdom of Great Britain and Northern I...,35-44 years old,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)","Employed, full-time","Hybrid (some remote, some in-person)",Engineering manager,Retail and Consumer Services,32.0,18.0,...,True,False,False,Yes,Unfavorable,True,True,False,False,True


### Questão 1
Realize um teste de hipótese que verifique se desenvolvedores que utilizam Python possuem remuneração média anual diferente daqueles que não utilizam Python. Realize o teste com confiança de 95%.

Dica: use \
dados_q1 = df.dropna(subset=["salary_usd_yearly", "uses_python"]).copy() \
para gerar uma cópia limpa de NaN nas colunas de interesse.

H0: μ(Python) = μ(Não_Python) \
Ha: μ(Python) ≠ μ(Não_Python) 

In [70]:
dados_q1 = df.dropna(subset=["salary_usd_yearly", "uses_python"]).copy()

python = dados_q1[dados_q1["uses_python"] == 1]["salary_usd_yearly"]
nao_python = dados_q1[dados_q1["uses_python"] == 0]["salary_usd_yearly"]

weightstats.ztest(python, nao_python)

(7.675955987585452, 1.641894980280461e-14)

Resposta: pValue < 0.05; rejeitamos a hipótese nula. Desenvolvedores que utilizam Python possuem remuneração média anual diferente daqueles que não utilizam Python

### Questão 2
Realize um teste de hipótese para investigar se a proporção de profissionais com remuneração acima da mediana é igual entre trabalhadores remotos e trabalhadores não remotos.

Utilize as variáveis:

```text
salary_above_median
is_remote

Lembre-se de limpar NaN nas colunas de interesse. Realize o teste com 95% de confiança.

H0: p(tr) - p(tnr) = 0 \
Ha: p(tr) - p(tnr) ≠ 0

In [71]:
dados_q2 = df.dropna(subset=["salary_above_median", "is_remote"]).copy()

tabela_q2 = dados_q2.groupby("is_remote")["salary_above_median"].agg(
    acima_mediana="sum",
    total="count",
    proporcao="mean"
)

tabela_q2.index = tabela_q2.index.map({False: "Não remoto", True: "Remoto"})
tabela_q2

,acima_mediana,total,proporcao
is_remote,,,
Não remoto,5662,12340,0.458833
Remoto,4965,8929,0.556053


In [72]:
remoto_acima = tabela_q2.loc["Remoto", "acima_mediana"]
remoto_total = tabela_q2.loc["Remoto", "total"]

nao_remoto_acima = tabela_q2.loc["Não remoto", "acima_mediana"]
nao_remoto_total = tabela_q2.loc["Não remoto", "total"]
proportion.test_proportions_2indep(remoto_acima,
                                   remoto_total,
                                   nao_remoto_acima,
                                   nao_remoto_total,
                                   method='score')


<class 'statsmodels.stats.base.HolderTuple'>
statistic = 13.994650800152016
pvalue = 1.6805308226400124e-44
compare = 'diff'
method = 'score'
variance = 4.826022038923162e-05
alternative = 'two-sided'
prop1_null = 0.49964737411255816
prop2_null = 0.49964737411255816
tuple = (13.994650800152016, 1.6805308226400124e-44)
diff = 0.09722024623207065
ratio = 1.2118858775174413
odds_ratio = 1.4772777498368395
value = 0

Resposta: Como pValue < 0.05, rejeitamos a hipótese nula. A proporção de trabalhadores com salário acima da mediana não é igual entre trabalhadores remotos e não remotos.

### Questão 3
Abóboras foram cultivadas sob duas condições experimentais, x e y. Do experimento x, uma amostra de 11 abóboras foi coletada e o desvio padrão amostral observado no peso das abóboras foi 0.8. Do experimento y, uma amostra de 9 abóboras foi coletada e o desvio padrão amostral observado foi 0.5. Assumindo que as distribuições de peso são normais, teste a hipótese de que as variâncias verdadeiras são iguais, contra a alternativa de que não são, ao nível de 90% de confiança.

H0​: σx2 ​= σy2 \
​H1​: σx2 ≠​ σy2

In [73]:
F_cal = (0.8**2)/(0.5**2)
F_cal

2.5600000000000005

Agora temos que encontrar o valor crítico à direita e à esquerda. Para tanto, usaremos a função isf da distribuição f, com os parâmetros 0.05 (alpha/2) e os dfs 10 e 8.

In [74]:
f.isf(0.05, 10, 8)

3.347163120233978

In [75]:
f.isf(0.95, 10, 8)

0.3255570361575732

Resposta: Como F_calc está entre o limite inferior e o limite superiot (0.32 < 2.56 < 3.34), falhamos em rejeitar a hipótese nula. Portanto, não há evidências estatísticas suficientes para concluir que as variâncias dos pesos das abóboras produzidas sob as condições experimentais X e Y sejam diferentes.

### Questão 4
Utilizando o dataset Ames.csv, realize um teste de hipótese que verifique a seguinte afirmação: "Casas com e sem lareira (Fireplace) têm o mesmo preço de venda (SalePrice)". Realize o teste com confiança de 95%.

H0: μfireplace = μno_fireplace \
Ha : μfireplace ≠ μno_fireplace

In [76]:
df = pd.read_csv('Ames.csv')
df.head(5)

,PID,GrLivArea,SalePrice,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,GeoRefNo,Prop_Addr,Latitude,Longitude
0,909176150,856,126000,30,RL,NaN,7890,Pave,NaN,Reg,...,NaN,0,3,2010,WD,Normal,909176150.0,436 HAYWARD AVE,42.018564,-93.651619
1,905476230,1049,139500,120,RL,42.0,4235,Pave,NaN,Reg,...,NaN,0,2,2009,WD,Normal,905476230.0,3416 WEST ST,42.024855,-93.663671
2,911128020,1001,124900,30,C (all),60.0,6060,Pave,NaN,Reg,...,NaN,0,11,2007,WD,Normal,911128020.0,320 S 2ND ST,42.021548,-93.614068
3,535377150,1039,114000,70,RL,80.0,8146,Pave,NaN,Reg,...,NaN,0,5,2009,WD,Normal,535377150.0,1524 DOUGLAS AVE,42.037391,-93.612207
4,534177230,1665,227000,60,RL,70.0,8400,Pave,NaN,Reg,...,NaN,0,11,2009,WD,Normal,534177230.0,2304 FILLMORE AVE,42.044554,-93.631818


In [77]:
no_fireplace = df[df['Fireplaces'] == 0]['SalePrice']
fireplace = df[df['Fireplaces'] > 0]['SalePrice']

In [78]:
stats.ttest_ind(no_fireplace, fireplace, equal_var=False)

TtestResult(statistic=-28.408052150857053, pvalue=1.4499262995182296e-149, df=2013.0087809518077)

Conclusão: Tendo em vista que pValue é muito pequeno (bem menor que 0.05), rejeitamos a hipótese nula. Portanto, casas com e sem lareira têm NÃO tem o mesmo preço de venda.

### Questão 5
Queremos testar a afirmação de que ouvir música clássica enquanto estuda faz com que os alunos concluam os trabalhos mais rapidamente. Pedimos a 10 alunos estudem em silêncio no primeiro semestre e estudem ouvindo música clássica no segundo semestre, e registramos o número médio de horas gasto executando trabalhos por semana em cada semestre. Os tempos gastos na execução de trabalhos em 10 semanas de estudo estão listados a seguir. Podemos concluir que estudar ouvindo música clássica reduz o número de horas gastas por semana na execução dos trabalhos escolares? Analise o valor de t_calc (statistic) com confiança de 95%.

Importante: Sua conclusão não deve utilizar o pvalue. Conclua utilizando o valor de t_calc e o t_crítico obtido da tabela t. Note que não é necessário fazer o cálculo à mão, já que o t_calc (statistic) é fornecido pelo Python.

Em silêncio: 14, 13, 16, 21, 15, 19, 11, 20, 19, 16 \
Ouvindo música: 12, 13, 15, 22, 16, 19, 8, 17, 18, 17

H0: μmusica >= μsilencio \
Ha : μmusica < μsilencio

In [80]:
p_silencio = [14, 13, 16, 21, 15, 19, 11, 20, 19, 16]
p_musica = [12, 13, 15, 22, 16, 19, 8, 17, 18, 17]
stats.ttest_rel(p_musica, p_silencio, alternative='less')

TtestResult(statistic=-1.412612867392256, pvalue=0.09570152734951828, df=9)

In [81]:
#valor pode ser obtido da tabela t, ou calculado conforme código a seguir
stats.t.ppf(0.95, df=9)

1.8331129326536333

Para alpha = 0,05 com cauda única e df=9, temos t_calc = 1.41 e t_crítico = 1.833. Portanto, falhamos em rejeitar a hipótese nula. Estudar com música não reduz o tempo de conclusão dos trabalhos.
O valor de pValue reafirma este resultado.

### Questão 6

Uma empresa desenvolveu um medicamento que supostamente aumenta o QI. É conhecido que o desvio padrão do QI na população em geral é 15 e o valor do QI tem distribuição normal. A empresa testa seu medicamento em 36 pacientes e obtém um QI médio de 97,65. Usando um valor alfa de 0.05, pode-se dizer que este valor de QI é significativamente diferente da média populacional, cujo valor é 100?
Realize o teste com confiança de 95%.

H0: μ = 100 \
Ha: μ ≠ 100

In [82]:
z_calc = (97.65 - 100) / (15/np.sqrt(36))
z_calc

-0.9399999999999977

In [83]:
norm.ppf(0.025)

-1.9599639845400545

In [84]:
norm.isf(0.025)

1.9599639845400545

z_calc = -0.94 está entre os valores críticos -1.96 e 1.96. Portanto, falhamos em rejeitar a hipótese nula. O QI médio da amostra é igual ao QI médio da população. O medicamento não altera o valor do QI.